Reading Data from HLS4ML LHC Jet dataset (https://zenodo.org/records/3602260)

In [18]:
import h5py
data=[]
fn_qcd = '/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_0_10000.h5'
with h5py.File(fn_qcd, 'r') as f:
    print("Keys in file:", list(f.keys()))
    for i in range(6):
        dataset_name = list(f.keys())[i]  # for example
        data = f[dataset_name][:]
        print(f"column name {dataset_name} and its shape {data.shape}")
# data

Keys in file: ['jetConstituentList', 'jetFeatureNames', 'jetImage', 'jetImageECAL', 'jetImageHCAL', 'jets', 'particleFeatureNames']
column name jetConstituentList and its shape (10000, 100, 16)
column name jetFeatureNames and its shape (59,)
column name jetImage and its shape (10000, 100, 100)
column name jetImageECAL and its shape (10000, 100, 100)
column name jetImageHCAL and its shape (10000, 100, 100)
column name jets and its shape (10000, 59)


Reading one file and preprocessing data

In [24]:
import h5py
import pandas as pd
import numpy as np

# Load file
data = h5py.File('/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_0_10000.h5', 'r')

# Extract jets and jet constituents
jets = data['jets'][:]                       # (10000, 59)
jetFeatureNames = data['jetFeatureNames'][:] # (59,)
constituents = data['jetConstituentList'][:] # (10000, 100, 16)
particleFeatureNames = data['particleFeatureNames'][:]

# Decode names (h5py gives bytes)
jetFeatureNames = [name.decode() for name in jetFeatureNames]
particleFeatureNames = [name.decode() for name in particleFeatureNames]

# ---- STEP 1: Get jet labels ----
# One-hot encoded in last 6 columns: j_g, j_q, j_w, j_z, j_t, j_undef
label_cols = [jetFeatureNames.index(n) for n in ["j_g", "j_q", "j_w", "j_z", "j_t", "j_undef"]]
jet_labels_onehot = jets[:, label_cols]
jet_labels = np.argmax(jet_labels_onehot, axis=1)  # integer labels

# ---- STEP 2: Extract only ptrel, etarot (for 200 columns total) ----
ptrel_idx  = particleFeatureNames.index("j1_ptrel")
etarot_idx = particleFeatureNames.index("j1_etarot")
rot_idx = particleFeatureNames.index("j1_etarot")


ptrel_all  = constituents[:, :, ptrel_idx]   # (10000, 100)
etarot_all = constituents[:, :, etarot_idx]  # (10000, 100)

# ---- STEP 3: Build DataFrame with 200 features ----
df_dict = {"jet_label": jet_labels}

for i in range(100):  # 100 constituents
    df_dict[f"ptrel_{i+1}"]  = ptrel_all[:, i]
    df_dict[f"etarot_{i+1}"] = etarot_all[:, i]

df = pd.DataFrame(df_dict)

# ---- STEP 4: Split into 6 datasets ----
jet_type_map = {
    0: "g",
    1: "q",
    2: "w",
    3: "z",
    4: "t",
    5: "undef"
}

datasets = {}
for label, name in jet_type_map.items():
    datasets[name] = df[df["jet_label"] == label].reset_index(drop=True)

# ---- Example: print sizes ----
for name, dset in datasets.items():
    print(f"{name}: {dset.shape}")


g: (2072, 201)
q: (1885, 201)
w: (2010, 201)
z: (1931, 201)
t: (2102, 201)
undef: (0, 201)


Reading all of data and preprocessing into one csv file for each jet type

In [ ]:
import h5py
import pandas as pd
import numpy as np
import glob
import os

# ---- SETTINGS ----
# input_dir = "/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train"
input_dirs = [
    "/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train",
    "/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/val"   # <-- second input dir
]

output_dir = "/nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/processed"
os.makedirs(output_dir, exist_ok=True)


# ---- STEP 1: Find all jetImage files ----
files = []
for d in input_dirs:
    files.extend(glob.glob(os.path.join(d, "jetImage*.h5")))

files = sorted(files)
print(f"Found {len(files)} files.")

# ---- STEP 2: Load and concatenate ----
all_dfs = []

for f in files:
    print(f"Processing {f}...")
    try:
        data = h5py.File(f, "r")
    except OSError as e:
        print(f"Skipping {f}, corrupted or incomplete ({e})")
        continue

    # ---- Skip files without required datasets ----
    if "jets" not in data or "jetConstituentList" not in data:
        print(f"Skipping {f}, missing 'jets' or 'jetConstituentList'")
        continue

    jets = data["jets"][:]                        # (N, ?)
    jetFeatureNames = [n.decode() for n in data["jetFeatureNames"][:]]

    constituents = data["jetConstituentList"][:]  # (N, 100, 16)
    particleFeatureNames = [n.decode() for n in data["particleFeatureNames"][:]]

    # ---- Labels ----
    label_names = ["j_g", "j_q", "j_w", "j_z", "j_t", "j_undef"]
    label_cols = []
    for n in label_names:
        if n in jetFeatureNames:
            label_cols.append(jetFeatureNames.index(n))
        else:
            print(f" {n} not found in {f}, filling zeros")
            label_cols.append(None)

    jet_labels_onehot = np.zeros((jets.shape[0], len(label_names)))
    for i, idx in enumerate(label_cols):
        if idx is not None:
            jet_labels_onehot[:, i] = jets[:, idx]
    jet_labels = np.argmax(jet_labels_onehot, axis=1)

    # ---- Features: ptrel, etarot, phirot ----
    required_features = ["j1_ptrel", "j1_etarot", "j1_phirot"]
    if not all(feat in particleFeatureNames for feat in required_features):
        print(f" Skipping {f}, missing one of {required_features}")
        continue

    ptrel_idx = particleFeatureNames.index("j1_ptrel")
    etarot_idx = particleFeatureNames.index("j1_etarot")
    phirot_idx = particleFeatureNames.index("j1_phirot")

    ptrel_all = constituents[:, :, ptrel_idx]   # (N, 100)
    etarot_all = constituents[:, :, etarot_idx] # (N, 100)
    phirot_all = constituents[:, :, phirot_idx] # (N, 100)

    # ---- Build dataframe: jet_label + 3*100 columns ----
    df_dict = {"jet_label": jet_labels}
    for i in range(100):  # up to 100 constituents
        df_dict[f"ptrel_{i+1}"] = ptrel_all[:, i]
        df_dict[f"etarot_{i+1}"] = etarot_all[:, i]
        df_dict[f"phirot_{i+1}"] = phirot_all[:, i]

    df = pd.DataFrame(df_dict)
    all_dfs.append(df)

# ---- STEP 3: Merge all ----
if not all_dfs:
    raise RuntimeError("No valid data found in any files!")

df_all = pd.concat(all_dfs, ignore_index=True)
print("Final shape of merged dataset:", df_all.shape)

# ---- STEP 4: Split into 6 datasets ----
labels_map = {0: "g", 1: "q", 2: "w", 3: "z", 4: "t", 5: "undef"}
for lbl, name in labels_map.items():
    df_type = df_all[df_all["jet_label"] == lbl]
    out_file = os.path.join(output_dir, f"jets_{name}.csv")
    df_type.to_csv(out_file, index=False)
    print(f"✅ Saved {name}: {df_type.shape} -> {out_file}")


Found 88 files.
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_0_10000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_10000_20000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_20000_30000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_30000_40000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_40000_50000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_50000_60000.h5...
❌ Skipping /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_50000_60000.h5, corrupted or incomplete (Unable to synchronously open file (truncated file: eof = 22069248, sblock->base_addr = 0, stored_eof = 47952254))
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_60000_70000.h5...
Processing /nlsasfs/home/mpplab-cdac/mitsa/Saurav/PH/train/jetImage_0_100p_70000_80000.h5...
Processing /nl